# 08 · Iterators & Generators

Data engineering means processing data that may not fit in memory. **Iterators**
and **generators** let you stream items one at a time instead of materializing
everything at once — the foundation of scalable pipelines.

## The iterator protocol

Anything you can loop over is **iterable**: calling `iter()` on it returns an
**iterator**, and `next()` pulls the next item until `StopIteration`. `for`
does all of this for you under the hood.

In [ ]:
nums = [10, 20, 30]
it = iter(nums)
print(next(it))
print(next(it))
print(next(it))
try:
    next(it)
except StopIteration:
    print('exhausted')

## Generators with `yield`

A function that uses `yield` becomes a **generator**: each `yield` produces a
value and *pauses*, resuming where it left off on the next request. State is
kept automatically. This reads like normal code but runs lazily.

In [ ]:
def countdown(n):
    while n > 0:
        yield n
        n -= 1

for x in countdown(3):
    print(x)

print('as list:', list(countdown(5)))

## Why lazy matters: process a big source with constant memory

The generator below yields one 'record' at a time. A pipeline built from
generators holds only the current item in memory, no matter how large the
source — the key to processing files bigger than RAM.

In [ ]:
def read_records(n):
    for i in range(n):
        yield {'id': i, 'amount': (i * 7) % 100}

# Chain lazy steps: filter -> transform -> aggregate, no big lists
records = read_records(1_000_000)
big = (r for r in records if r['amount'] >= 50)
taxed = (r['amount'] * 1.2 for r in big)
total = sum(taxed)          # everything streams; memory stays tiny
print('total:', round(total, 2))

## `itertools` — batteries for iterators

The standard library `itertools` module has fast, memory-efficient building
blocks. A few you'll reach for in data work:

In [ ]:
import itertools

# islice: take the first N from any (even infinite) iterator
first5 = list(itertools.islice(range(1_000_000), 5))
print('islice:', first5)

# chain: concatenate iterables lazily
print('chain:', list(itertools.chain([1, 2], [3, 4])))

# groupby: group CONSECUTIVE items by a key (sort first!)
rows = [('US', 1), ('US', 2), ('GB', 3), ('GB', 4), ('DE', 5)]
for country, group in itertools.groupby(rows, key=lambda r: r[0]):
    print(country, [g[1] for g in group])

## Batching a stream (a real DE pattern)

Loaders often write in batches (e.g. 1000 rows per INSERT). Here's a reusable
generator that chunks any iterable — you'll use this idea in the capstone.

In [ ]:
import itertools

def batched(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(itertools.islice(it, size))
        if not chunk:
            return
        yield chunk

for batch in batched(range(10), 4):
    print('batch of', len(batch), '->', batch)

### Recap

Iterables give iterators via `iter()`; `next()` advances them; `yield` makes
lazy generators that stream with constant memory; `itertools` supplies
`islice`, `chain`, `groupby`; batching is a core loader pattern. Next: modules
and the standard library.